# RASCALS_IBM Z Datathon 2026 Project
## ContinuityGuard: 03 - Engineering audit and platform proof

ANBU SELVAN T; SAHANA G; DHARSHINI E; POOJA V

Hindustan Institute of Technology and Science

This notebook tests the hardened source on LinuxONE, exercises actual local authenticated HTTP ingestion, and verifies native Granite evidence generation. Synthetic fixtures are explicitly labeled. It does not turn a reproducibility rerun into a new independent model evaluation.

In [ ]:
from pathlib import Path
import os, sys, platform, json
ROOT = Path.cwd()
if not (ROOT / 'continuityguard').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'continuityguard').is_dir(), 'Open this notebook from the project notebooks folder'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Team: RASCALS')
print('Architecture:', platform.machine())
print('Python:', platform.python_version())
print('Project:', ROOT)
assert platform.machine() == 's390x', 'This submission notebook is intended to run on LinuxONE'


In [ ]:
import unittest, io, time, importlib.metadata
suite = unittest.defaultTestLoader.discover(str(ROOT/'tests'))
stream = io.StringIO()
started = time.perf_counter()
result = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)
print(stream.getvalue())
correctness = {'architecture': platform.machine(), 'python': platform.python_version(), 'tests_run': result.testsRun, 'failures':len(result.failures), 'errors':len(result.errors), 'skipped':len(result.skipped), 'successful':result.wasSuccessful(), 'elapsed_seconds':round(time.perf_counter()-started,2), 'output':stream.getvalue()}
(ROOT/'artifacts/correctness-report.json').write_text(json.dumps(correctness,indent=2))
assert result.wasSuccessful()


### Actual HTTP integration
A separate loopback app and temporary state receive synthetic fixture events via the real HTTP endpoint, bounded queue and SQLite path. The benchmark does not reset the demo app. Queue-ack timing is distinct from inference timing. No WAN or browser latency claim follows.

In [ ]:
from continuityguard.http_benchmark import main as benchmark_http
http_result = benchmark_http(ROOT)
display(http_result)

### Local predictive + generative proof
The predictive model creates a sandbox incident. Local IBM Granite selects a verified explanation under a strict schema. Every evidence ID cited in its text must be included. Missing/invalid generation has a labeled deterministic fallback; this cell requires real local generation to pass.

In [ ]:
from continuityguard.runtime import Engine
from continuityguard.explain import Explainer
from continuityguard.simulation import Experiment, simulate
import tempfile
events,_=simulate(Experiment('pool_exhaustion',101,strength=1.12))
with tempfile.TemporaryDirectory() as folder:
    engine=Engine(ROOT/'artifacts',str(Path(folder)/'proof.sqlite'))
    incidents=[]
    for k in range(0,len(events),4):
        incidents.extend(engine.ingest(events[k:k+4]))
        if incidents:break
    assert incidents
    incident=incidents[0]
    explainer=Explainer('/home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf',engine.journal)
    explainer.request(incident)
    explainer.close()
    explanation=explainer.status(incident['id'])
    assert explanation['status']=='complete'
    assert explanation['mode']=='local Granite · constrained generation',explanation
    assert explanation['evidence_ids']==[e['id'] for e in incident['evidence']]
    proof={'architecture':platform.machine(),'model_manifest':engine.bundle.manifest,'predictive_incident':incident,'generative_explanation':explanation,'audit':engine.journal.verify(),'scope':'Actual native s390x predictive and local generative execution; synthetic incident; constrained wording, not open-ended root-cause reasoning.'}
    (ROOT/'evidence/jupyter-ai-proof.json').write_text(json.dumps(proof,indent=2))
    display(proof)
    engine.close()
